# Лабораторная работа №2. Кластеризация данных

In [10]:
import matplotlib.pyplot as plt
from sklearn.cluster import MiniBatchKMeans
from scipy.spatial.distance import euclidean
from sklearn import metrics
from sklearn.cluster import KMeans, AgglomerativeClustering, AffinityPropagation, SpectralClustering
import numpy as np
import pandas as pd

Считайте банковские данные из файла DATASET FOR CASE.csv в память в виде объекта Pandas.DataFrame (разделитель между колонками - табуляция, кодировка - ansi, для разделения целой и дробной частей чисел используется запятая). Выведите информацию о датасете, поймите и опишите его структуру. TARGET (take a credit) - целевая бинарная переменная, информирующая о выдаче / невыдаче кредита.

In [11]:
# Ваш код здесь
#указаны последовательно путь к файлу(filepath_or_buffer), разделитель между колонками(sep),
#кодировка(encoding), разделитель целой и дробной части(decimal)
data=pd.read_csv(filepath_or_buffer='DATASET FOR CASE.csv',sep='\t',encoding='ansi',decimal=',')
data.dropna()
#Ограничение по возрасту
data=data[data['Age of client']>0]
X, y = data.iloc[:,:-1].copy(), data["TARGET (take a credit)"].copy()
print(data.info())
print(data.shape)
mins=[0]*23
maxs=[0]*23
for i in range(data.shape[1]):
    mins[i]=data.iloc[:,i].min()
    maxs[i]=data.iloc[:,i].max()
print(mins)
print(maxs)
#В датасете имеется 20 параметров, каждый из которых характеризует клиента банка. Среди них:
#размер последнего кредита, возраст, количество кредитов с 2010, количество транзакций за месяц и расстояние от дома клиента до офиса банка. Все данные в числовом формате.
print(data)

<class 'pandas.core.frame.DataFrame'>
Int64Index: 48741 entries, 0 to 48751
Data columns (total 23 columns):
 #   Column                                                                                 Non-Null Count  Dtype  
---  ------                                                                                 --------------  -----  
 0   Number of calculations of credit limit                                                 48741 non-null  float64
 1   Amount of the last loan in the bank                                                    48741 non-null  float64
 2   Number of credit in the bank from 2010 year
                                           48741 non-null  float64
 3   Months from closing of last loan                                                       48741 non-null  float64
 4   Number of credits in the bank                                                          48741 non-null  float64
 5   Size of payments for last creidts                                         

Представим, что значение переменной TARGET (take a credit) неизвестно. С помощью kmeans и метода локтя определите количество кластеров для датасета. 

In [12]:
# Ваш код здесь
#создаем словарь для хранения суммы квадратов расстояний до ближайшего центра кластера
wws={}
for k in range(1, 20):
    #применяем кластеризацию
    kmeans = KMeans(n_clusters=k).fit(data.iloc[:,:-1])
    #сохраняем сумму квадратов отклонений
    wws[k] = kmeans.inertia_
    if k==1:
        pass
    else:
        sil_coeff = metrics.silhouette_score(X, kmeans.labels_)
        print(str(k)+':  '+str(sil_coeff))

2:  0.6021357272050781
3:  0.5685969512563089
4:  0.5365631595143022
5:  0.4460116150132276
6:  0.4463752311742982
7:  0.4735800567380212
8:  0.47665559210194275
9:  0.4172810087827776
10:  0.41610805043309174
11:  0.4017582075778422
12:  0.3995368117220761
13:  0.39984505663955955
14:  0.402831328902954
15:  0.4013817211358702
16:  0.3698237016506382
17:  0.3539536645074628
18:  0.36597512175174385
19:  0.3474979584554835


In [ ]:
#подключаем бибилиотеки для plotly
import plotly
import plotly.graph_objs as go
import plotly.express as px
from plotly.subplots import make_subplots
fig = go.Figure()
fig.add_trace(go.Scatter(x=list(wws.keys()), y=list(wws.values()), name='WWS'))
fig.update_layout(xaxis_title="k",yaxis_title="WWS",legend_orientation="h")
fig.update_traces(hoverinfo="all", hovertemplate="Аргумент: %{x}<br>Функция: %{y}")
fig.show()

Используя kmeans и выше определенное число кластеров k, получите метки классов, выведите их и добавьте их в исходный датасет, записав его под новым именем.

In [ ]:
# Ваш код здесь
kmeans = KMeans(n_clusters=3).fit(data)
#добавляем метки классов
extended_data=data.copy()
extended_data["clusters"] = kmeans.labels_
#выводим метки классов
print(extended_data["clusters"].sum())

11105


Примените для решения той же задачи агломеративную, аффинную и спектральную кластеризации: инструменты AgglomerativeClustering, AffinityPropagation, SpectralClustering из sklearn.cluster, и, используя метрики качества кластеризации, прокомментируйте достигнутые результаты.

In [ ]:
# Ваш код здесь
#недостаточно оперативной памяти для возможности работы с данным методом
algo=AffinityPropagation()

algo.fit(X)
d.append(({
    'ARI': metrics.adjusted_rand_score(y, algo.labels_),
    'AMI': metrics.adjusted_mutual_info_score(y, algo.labels_),
    'Homogenity': metrics.homogeneity_score(y, algo.labels_),
    'Completeness': metrics.completeness_score(y, algo.labels_),
    'V-measure': metrics.v_measure_score(y, algo.labels_),
    'Silhouette': metrics.silhouette_score(X, algo.labels_)}))     

MemoryError: Unable to allocate 17.7 GiB for an array with shape (48741, 48741) and data type float64

In [18]:
#массив для набора метрик
d = []
for k in range(2,9):
    algo=AgglomerativeClustering(n_clusters=k)
    algo.fit(X)
    d.append(({
        'ARI': metrics.adjusted_rand_score(y, algo.labels_),
        'AMI': metrics.adjusted_mutual_info_score(y, algo.labels_),
        'Homogenity': metrics.homogeneity_score(y, algo.labels_),
        'Completeness': metrics.completeness_score(y, algo.labels_),
        'V-measure': metrics.v_measure_score(y, algo.labels_),
        'Silhouette': metrics.silhouette_score(X, algo.labels_)}))

results = pd.DataFrame(data=d, columns=['ARI', 'AMI', 'Homogenity',
                                           'Completeness', 'V-measure', 
                                           'Silhouette'],
                       index=['Agglomerative 2', 'Agglomerative 3',
                       'Agglomerative 4','Agglomerative 5','Agglomerative 6',
                       'Agglomerative 7','Agglomerative 8'])

print(results)

In [ ]:
d = []
for k in range(2,9):
    algo=SpectralClustering(n_clusters=k, random_state=1, affinity='nearest_neighbors')
    algo.fit(X)
    d.append(({
        'ARI': metrics.adjusted_rand_score(y, algo.labels_),
        'AMI': metrics.adjusted_mutual_info_score(y, algo.labels_),
        'Homogenity': metrics.homogeneity_score(y, algo.labels_),
        'Completeness': metrics.completeness_score(y, algo.labels_),
        'V-measure': metrics.v_measure_score(y, algo.labels_),
        'Silhouette': metrics.silhouette_score(X, algo.labels_)}))

results = pd.DataFrame(data=d, columns=['ARI', 'AMI', 'Homogenity',
                                           'Completeness', 'V-measure', 
                                           'Silhouette'],
                       index=['Spectral 2', 'Spectral 3',
                       'Spectral 4','Spectral 5','Spectral 6',
                       'Spectral 7','Spectral 8'])

print(results)

                 ARI       AMI  Homogenity  Completeness  V-measure  \
Spectral 2 -0.002076  0.000100    0.000247      0.000082   0.000123   
Spectral 3 -0.001197  0.000302    0.000733      0.000226   0.000346   
Spectral 4 -0.002058  0.000245    0.000849      0.000178   0.000294   
Spectral 5 -0.003261  0.001046    0.003852      0.000641   0.001099   
Spectral 6 -0.005281  0.001079    0.004388      0.000654   0.001139   
Spectral 7 -0.005909  0.001222    0.005204      0.000737   0.001291   
Spectral 8 -0.006751  0.001222    0.005563      0.000735   0.001299   

            Silhouette  
Spectral 2    0.464397  
Spectral 3    0.442581  
Spectral 4    0.282335  
Spectral 5    0.324996  
Spectral 6    0.243565  
Spectral 7    0.250408  
Spectral 8    0.193723  
